In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import numpy as np
from pathlib import Path
import time
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
import seaborn as sns


In [5]:
from model import IntentClassifier, BANKING77_CONFIG

In [6]:
BANKING77_LABELS = [
    "activate_my_card",
    "age_limit",
    "apple_pay_or_google_pay",
    "atm_support",
    "automatic_top_up",
    "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit",
    "beneficiary_not_allowed",
    "cancel_transfer",
    "card_about_to_expire",
    "card_acceptance",
    "card_arrival",
    "card_delivery_estimate",
    "card_linking",
    "card_not_working",
    "card_payment_fee_charged",
    "card_payment_not_recognised",
    "card_payment_wrong_exchange_rate",
    "card_swallowed",
    "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised",
    "change_pin",
    "compromised_card",
    "contactless_not_working",
    "country_support",
    "declined_card_payment",
    "declined_cash_withdrawal",
    "declined_transfer",
    "direct_debit_payment_not_recognised",
    "disposable_card_limits",
    "edit_personal_details",
    "exchange_charge",
    "exchange_rate",
    "exchange_via_app",
    "extra_charge_on_statement",
    "failed_transfer",
    "fiat_currency_support",
    "get_disposable_virtual_card",
    "get_physical_card",
    "getting_spare_card",
    "getting_virtual_card",
    "lost_or_stolen_card",
    "lost_or_stolen_phone",
    "order_physical_card",
    "passcode_forgotten",
    "pending_card_payment",
    "pending_cash_withdrawal",
    "pending_top_up",
    "pending_transfer",
    "pin_blocked",
    "receiving_money",
    "Refund_not_showing_up",
    "request_refund",
    "reverted_card_payment?",
    "supported_cards_and_currencies",
    "terminate_account",
    "top_up_by_bank_transfer_charge",
    "top_up_by_card_charge",
    "top_up_by_cash_or_cheque",
    "top_up_failed",
    "top_up_limits",
    "top_up_reverted",
    "topping_up_by_card",
    "transaction_charged_twice",
    "transfer_fee_charged",
    "transfer_into_account",
    "transfer_not_received_by_recipient",
    "transfer_timing",
    "unable_to_verify_identity",
    "verify_my_identity",
    "verify_source_of_funds",
    "verify_top_up",
    "virtual_card_not_working",
    "visa_or_mastercard",
    "why_verify_identity",
    "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal",
]


In [7]:
assert len(BANKING77_LABELS) == 77
assert len(set(BANKING77_LABELS)) == 77

ID2LABEL = {i: label for i, label in enumerate(BANKING77_LABELS)}

In [8]:
TRAIN_CONFIG = {
    "batch_size"   : 64,
    "epochs"       : 20, # 30 -> 20,we  were plateauing at 17-18 anyway
    "lr"           : 1e-4,
    "weight_decay" : 1e-2,
    "grad_clip"    : 1.0,
    "num_classes"  : 77,
    "pool"         : "mean",
    "seed"         : 42,
    "save_path"    : "best_model.pt",
    "train_emb"    : r"D:/@FYP-IntentClassifier/text_preprocessing/train_embeddings.pt",
    "val_emb"      : r"D:/@FYP-IntentClassifier/text_preprocessing/val_embeddings.pt",
    "test_emb"     : r"D:/@FYP-IntentClassifier/text_preprocessing/test_embeddings.pt",
}

In [9]:
class EmbeddingDataset(Dataset):
    """
    Expects a .pt file containing a dict:
        {
            "embeddings"     : Tensor[N, seq_len, emb_dim],
            "attention_mask" : Tensor[N, seq_len],   # 1 = real token, 0 = pad
            "labels"         : Tensor[N],
        }
    """
    def __init__(self, path: str):
        data = torch.load(path, weights_only=True)
        self.embeddings      = data["embeddings"]       # (N, seq_len, emb_dim)
        self.attention_mask  = data["attention_mask"]   # (N, seq_len)
        self.labels          = data["labels"]           # (N,)
 
    def __len__(self):
        return len(self.labels)
 
    def __getitem__(self, idx):
        return (
            self.embeddings[idx],
            self.attention_mask[idx],
            self.labels[idx],
        )
 

HELPERS

In [10]:
def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
 

In [ ]:
def accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    preds = logits.argmax(dim=-1)
    return (preds == labels).float().mean().item()



In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for emb, mask, labels in loader:
        emb, mask, labels = emb.to(device), mask.to(device), labels.to(device)
        logits = model(emb, mask)
        total_loss += criterion(logits, labels).item() * len(labels)
        total_acc  += accuracy(logits, labels)         * len(labels)
        n          += len(labels)
    return total_loss / n, total_acc / n

'''This function runs the model on all batches in a validation/test set 
and returns the overall average loss and accuracy without performing any training.
'''
 
 

In [42]:
@torch.no_grad()
def predict(model, loader, device, id2label=None, threshold=0.4, temp=2.0, intent_thresholds=None):
    """
    Run inference and print confidence score + predicted intent per sample.
 
    Args:
        threshold : samples with max-softmax below this are marked REJECTED
        temp      : temperature for scaling logits (1.0 = no scaling)
                    tune this after training with compute_ece()
    """
    model.eval()
    results = []
    for emb, mask, labels in loader:
        emb, mask = emb.to(device), mask.to(device)
        logits = model(emb, mask)
        probs  = torch.softmax(logits / temp, dim=-1)
        confidence, predicted = probs.max(dim=-1)
 
        for i in range(len(labels)):
            conf  = confidence[i].item()
            pred  = predicted[i].item()
            true  = labels[i].item()
            label = id2label[pred] if id2label else str(pred)
            
            if intent_thresholds:
                thresh = intent_thresholds.get(label, threshold)
            else:
                thresh = threshold

            results.append({
                "predicted_id"    : pred,
                "predicted_label" : label,
                "confidence"      : round(conf, 4),
                "true_id"         : true,
                "correct"         : pred == true,
                "rejected"        : conf < thresh,      # ← now per-intent
                "threshold_used"  : round(thresh, 4),   # ← useful for debugging
            })
            
    print(f"\n{'STATUS':<32}{'CONFIDENCE':>12}  {'CORRECT':>8}  TRUE ID")
    print("-" * 70)
    for r in results[:20]:                          # show first 20
        if r["rejected"]:
            status = "REJECTED"
        else:
            status = r["predicted_label"]
        correct_str = "Y" if r["correct"] else "N"
        print(f"{status:<32}{r['confidence']:>12.4f}  {correct_str:>8}  {r['true_id']}")
 
    n_total    = len(results)
    n_rejected = sum(r["rejected"] for r in results)
    n_correct  = sum(r["correct"] and not r["rejected"] for r in results)
    print(f"\nTotal: {n_total}  |  Rejected: {n_rejected}  |  "
          f"Accepted correct: {n_correct}/{n_total - n_rejected}")
    return results

In [43]:
def compute_ece(results, n_bins=15):
    """
    Expected Calibration Error — measures how well confidence matches accuracy.
    Lower is better. A perfectly calibrated model has ECE = 0.
 
    Prints a reliability breakdown and returns the ECE scalar.
    """
    confidences = np.array([r["confidence"] for r in results])
    correctness = np.array([r["correct"]    for r in results], dtype=float)
 
    bins    = np.linspace(0, 1, n_bins + 1)
    ece     = 0.0
    n_total = len(results)
 
    print(f"\n{'BIN':>14}  {'COUNT':>6}  {'AVG CONF':>10}  {'ACCURACY':>10}  {'|DIFF|':>8}")
    print("-" * 58)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        avg_conf = confidences[mask].mean()
        avg_acc  = correctness[mask].mean()
        weight   = mask.sum() / n_total
        diff     = abs(avg_conf - avg_acc)
        ece     += weight * diff
        print(f"({lo:.2f}, {hi:.2f}]  {mask.sum():>6}  {avg_conf:>10.4f}  "
              f"{avg_acc:>10.4f}  {diff:>8.4f}")
 
    print(f"\nECE = {ece:.4f}  (0 = perfect, >0.05 = needs calibration)")
    return ece

In [44]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = float("inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

In [ ]:
def train(cfg: dict = TRAIN_CONFIG):
    set_seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")
 
    # ── Data ──────────────────────────────────────────────────────────────────
    train_ds = EmbeddingDataset(cfg["train_emb"])
    val_ds = EmbeddingDataset(cfg["val_emb"])
    test_ds  = EmbeddingDataset(cfg["test_emb"])
 
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )
    
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg["batch_size"] * 2,
        shuffle=False,
        num_workers=0, #Windows issue
        pin_memory=True,
    )
 
    # ── Model ─────────────────────────────────────────────────────────────────
    model = IntentClassifier(
        cfg=BANKING77_CONFIG,
        num_classes=cfg["num_classes"],
        pool=cfg["pool"],
    ).to(device)
 
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}\n")
 
    # ── Optimiser & scheduler ─────────────────────────────────────────────────
    optimizer = AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    ) 
    ##
 
    # OneCycleLR: warmup + cosine decay in one shot — works well for transformers
    scheduler = OneCycleLR(
        optimizer,
        max_lr=cfg["lr"],
        steps_per_epoch=len(train_loader),
        epochs=cfg["epochs"],
        pct_start=0.1,          # 10 % of steps = warmup
        anneal_strategy="cos",
    )
 
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
 
    # ── Loop ──────────────────────────────────────────────────────────────────
    best_acc   = 0.0
    history    = []
    early_stopping = EarlyStopping(patience=5)
 
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        epoch_loss, epoch_acc, n = 0.0, 0.0, 0
        t0 = time.time()
 
        for emb, mask, labels in train_loader:
            emb, mask, labels = emb.to(device), mask.to(device), labels.to(device)
 
            optimizer.zero_grad()
            logits = model(emb, mask)
            loss   = criterion(logits, labels)
            loss.backward()
 
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
 
            optimizer.step()
            scheduler.step()
 
            bs = len(labels)
            epoch_loss += loss.item() * bs
            epoch_acc  += accuracy(logits, labels) * bs
            n          += bs
 
        train_loss = epoch_loss / n
        train_acc  = epoch_acc  / n
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - t0
 
        history.append({
            "epoch": epoch,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss,     "val_acc": val_acc,
        })
 
        flag = ""
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), cfg["save_path"])
            flag = "  ← saved"
 
        print(
            f"Epoch {epoch:02d}/{cfg['epochs']}  "
            f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
            f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
            f"({elapsed:.1f}s){flag}"
        )
        
        early_stopping.step(val_loss)          # ADDED THIS
        if early_stopping.should_stop:         # ADDED THIS
            print(f"\nEarly stopping at epoch {epoch}")
            break
 
    print(f"\nBest val accuracy: {best_acc:.4f}")
    return history


In [46]:
# ── Entry point ───────────────────────────────────────────────────────────────
 
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    # ── Train ──────────────────────────────────────────────────────────────────
    history = train()
 
    # ── Load best checkpoint ───────────────────────────────────────────────────
    model = IntentClassifier(
        cfg=BANKING77_CONFIG,
        num_classes=TRAIN_CONFIG["num_classes"],
        pool=TRAIN_CONFIG["pool"],
    ).to(device)
    model.load_state_dict(torch.load(TRAIN_CONFIG["save_path"], map_location=device))
 
    test_ds = EmbeddingDataset(TRAIN_CONFIG["test_emb"])
    test_loader = DataLoader(
        test_ds,
        batch_size=TRAIN_CONFIG["batch_size"] * 2,
        shuffle=False,
        num_workers=0,
    )
 
    # ── Confidence + intent predictions ───────────────────────────────────────
    print("\n" + "=" * 70)
    print("PREDICTIONS ")
    print("=" * 70)
    results = predict(
        model, test_loader, device,
        id2label=ID2LABEL,
        threshold=0.4,
        temp=1.0,           # set T > 1 if ECE shows overconfidence
    )
 
    # ── Calibration error ──────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("CALIBRATION  (ECE)")
    print("=" * 70)
    ece = compute_ece(results)

Device: cuda

Trainable parameters: 14,825,549

Epoch 01/20  train_loss=4.2325  train_acc=0.0565  val_loss=3.6993  val_acc=0.1495  (33.2s)  ← saved
Epoch 02/20  train_loss=2.7318  train_acc=0.4036  val_loss=1.8415  val_acc=0.6669  (33.0s)  ← saved
Epoch 03/20  train_loss=1.6249  train_acc=0.7502  val_loss=1.4434  val_acc=0.8224  (33.2s)  ← saved
Epoch 04/20  train_loss=1.3407  train_acc=0.8425  val_loss=1.3781  val_acc=0.8558  (33.5s)  ← saved
Epoch 05/20  train_loss=1.2137  train_acc=0.8820  val_loss=1.3264  val_acc=0.8879  (33.3s)  ← saved
Epoch 06/20  train_loss=1.1145  train_acc=0.9191  val_loss=1.2835  val_acc=0.8885  (33.3s)  ← saved
Epoch 07/20  train_loss=1.0660  train_acc=0.9289  val_loss=1.2849  val_acc=0.8905  (33.4s)  ← saved
Epoch 08/20  train_loss=1.0095  train_acc=0.9480  val_loss=1.2392  val_acc=0.8999  (33.4s)  ← saved
Epoch 09/20  train_loss=0.9731  train_acc=0.9599  val_loss=1.2387  val_acc=0.9072  (33.5s)  ← saved
Epoch 10/20  train_loss=0.9410  train_acc=0.9702  va